# Gash (Templar)

Gash is a Templar melee nuke. The caster cuts themselves for **2.5% of current HP** to deliver
one heavy, weapon-scaled blow to an adjacent target. It is the Templar's main "spend HP for
damage" button. Its damage scales with both the caster's **weapon value (WV)** and their
**modified Gash skill**.

**Rebalanced.** The bonus used to be `clamp(3p, 30, 600)`. It reached its +600% ceiling at
modified skill 200, so the last third of a top player's skill did nothing. It now uses the shared
mod-300 knee, `skill_knee_300(p, 0, 600)` with a +30% floor (`gash_bonus_pct`). The bonus reaches
+600% at modified skill 300 and keeps growing at half that slope, with no ceiling. The slope below
300 is lower than before, so Gash hits softer at the raise cap. The "before" curves below show the
old formula.

| | |
|---|---|
| Class | Templar |
| Target | Adjacent enemy (Chebyshev distance 1) |
| Cost | `max(1, a_hp/40)` milli-HP, i.e. 2.5% of **current** HP; refused if that would kill |
| Cooldown | 10 s |
| Damage | `W·(100 + bonus%)/100 + rand(3W)`, with `bonus% = max(30, skill_knee_300(p, 0, 600))` |
| Damage type | 0 (melee, `(dam − armor) × ¼`) |
| Source | `server/src/driver/skill/templar.rs::skill_gash`, `gash_bonus_pct`; knee in `driver/skill/common.rs::skill_knee_300` |

---

# Formula

Let $W = \max(1, \text{WV})$ be the caster's weapon value and $p$ the modified Gash skill. Gash
uses $p$ directly, with no immunity or race modifier. The knee from 0 to +600% over 300 points is
exactly $2p$, and half that slope beyond 300 is exactly $+1\%$ per point, so there is no rounding:

$$
\text{bonus}\% = \begin{cases} \max(30,\ 2p) & p \le 300 \\ 300 + p & p > 300 \end{cases}
\qquad
\text{dam} = \left\lfloor \frac{W\,(100 + \text{bonus}\%)}{100} \right\rfloor + \text{rand}(3W)
$$

where $\text{rand}(n)$ is uniform over $0 \dots n-1$. The hit is passed to `do_hurt` as type 0, so
the target takes

$$
\text{HP lost} = \frac{\max(0,\ \text{dam} - \text{armor}) \times 250}{1000}
$$

| Modified skill | bonus % | Fixed part | Before: bonus % | Notes |
|---|---|---|---|---|
| 1 – 15 | 30 | $1.3W$ | 30 – 45 | floor (before: floor to 10) |
| 105 | 210 | $3.1W$ | 315 | raise cap |
| 150 | 300 | $4W$ | 450 | |
| 200 | 400 | $5W$ | 600 | old ceiling |
| 300 | 600 | $7W$ | 600 | knee |
| 350 | 650 | $7.5W$ | 600 | |
| 450 | 750 | $8.5W$ | 600 | no ceiling |

The random part, $\text{rand}(3W)$, averages $1.5W$ at every skill level. Expected raw damage
before armor is therefore $W\,(1 + \text{bonus}\%/100) + 1.5W - 0.5$.

---

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import mag_model as mm

mm.apply_style()
P = mm.SKILL_RANGE

# Model

In [ ]:
def gash_bonus_pct(p):
    return np.maximum(mm.knee_300(p, 0, 600), 30)


# Before the rebalance: linear 3p, clamped to 30..600 (ceiling at skill 200).
def old_gash_bonus_pct(p):
    return mm.clamp(np.asarray(p) * 3, 30, 600)


def gash_fixed(p, wv, bonus_pct=gash_bonus_pct):
    w = np.maximum(1, wv)
    return mm.idiv(w * (100 + bonus_pct(p)), 100)


def gash_expected_hp(p, wv, armor=0, bonus_pct=gash_bonus_pct):
    """Exact expected HP removed from the target by one Gash."""
    w = np.maximum(1, wv)
    return mm.expected_hurt_uniform(gash_fixed(p, wv, bonus_pct), 3 * w, armor, kind=0)


def old_gash_fixed(p, wv):
    return gash_fixed(p, wv, old_gash_bonus_pct)


def old_gash_expected_hp(p, wv, armor=0):
    return gash_expected_hp(p, wv, armor, old_gash_bonus_pct)


def gash_self_cost_hp(current_hp):
    """HP the caster loses: `max(1, a_hp/40)` milli-HP."""
    a_hp = np.asarray(current_hp) * 1000
    return np.maximum(1, mm.idiv(a_hp, 40)) / 1000


def melee_expected_hp(wv, strength, armor=0):
    """Plain auto-attack on a normal hit (die >= 3): WV + rand(6)+1 + rand(str/2), no bonus."""
    # Sum of two independent uniforms; enumerate exactly for the armor clamp.
    a = np.arange(6) + 1
    b = np.arange(max(1, strength // 2)) if strength > 3 else np.array([0])
    dam = wv + a[:, None] + b[None, :]
    return mm.hurt_hp(dam, armor, 0).mean()

# Validation

- `templar_bonuses_reach_their_design_value_at_skill_300`: `gash_bonus_pct` gives +30% at
  skill 0, +300% at 150, +600% at 300 and +750% at 450.

The cells below also check the floor, the old formula, and the closed-form expectation against
brute-force enumeration of every die roll.

In [ ]:
assert [int(gash_bonus_pct(p)) for p in (0, 150, 300, 450)] == [30, 300, 600, 750]
assert gash_bonus_pct(15) == 30 and gash_bonus_pct(16) == 32
assert gash_bonus_pct(105) == 210 and gash_bonus_pct(350) == 650
assert np.array_equal(gash_bonus_pct(P), np.where(P <= 300, np.maximum(2 * P, 30), 300 + P))
assert gash_fixed(100, 40) == 120          # 40 * 300 / 100
assert old_gash_bonus_pct(30) == 90 and old_gash_bonus_pct(150) == 450
assert old_gash_bonus_pct(200) == 600 and old_gash_bonus_pct(400) == 600

for p, wv, armor in [(1, 1, 0), (50, 25, 30), (120, 40, 80), (200, 60, 0), (5, 3, 40), (450, 60, 40)]:
    w = max(1, wv)
    brute = np.mean([mm.hurt_hp(gash_fixed(p, wv) + k, armor, 0) for k in range(3 * w)])
    assert np.isclose(gash_expected_hp(p, wv, armor), brute), (p, wv, armor)

assert gash_self_cost_hp(400) == 10.0       # 2.5% of 400 HP
assert gash_self_cost_hp(0.02) == 0.001     # floor of 1 milli-HP
print("Gash model OK")

# Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for wv in mm.WV_SWEEP:
    line, = ax.plot(P, gash_expected_hp(P, wv, armor=0), label=f"WV {wv}")
    if wv == 40:
        ax.plot(P, old_gash_expected_hp(P, wv, armor=0), color=line.get_color(), ls=":", lw=1.2,
                label=f"before, WV {wv}")
mm.skill_axis(ax, "expected HP per Gash", "Expected damage vs skill (armor 0)")
mm.mark_caps(ax, extra=[(200, "old ceiling"), (300, "knee")])
ax.legend()

ax = axes[1]
for armor in mm.ARMOR_SWEEP:
    line, = ax.plot(P, gash_expected_hp(P, 40, armor=armor), label=f"armor {armor}")
    if armor == 20:
        ax.plot(P, old_gash_expected_hp(P, 40, armor=armor), color=line.get_color(), ls=":", lw=1.2,
                label=f"before, armor {armor}")
mm.skill_axis(ax, "expected HP per Gash", "Expected damage vs skill (WV 40)")
mm.mark_caps(ax, extra=[(200, "old ceiling"), (300, "knee")])
ax.legend()
plt.tight_layout()

The fixed part grows linearly, by $0.03W$ per skill point, all the way to skill 200. Only the
minimum (1.3W) and maximum (7W) are clamped. Armor shifts every curve down by a near-constant
$\text{armor}/4$ HP. The $\text{rand}(3W)$ term is wide, so even WV 40 against armor 80 still
averages ~9 HP at the skill floor; a Gash rarely glances off completely.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Range of the roll: min/max/mean raw damage at WV 40.
ax = axes[0]
wv = 40
lo = gash_fixed(P, wv)
ax.fill_between(P, mm.hurt_hp(lo, 0), mm.hurt_hp(lo + 3 * wv - 1, 0), alpha=0.25, label="min–max roll")
line, = ax.plot(P, gash_expected_hp(P, wv), label="expected")
ax.plot(P, old_gash_expected_hp(P, wv), color=line.get_color(), ls=":", lw=1.2, label="before, expected")
for strength in (40, 100):
    ax.axhline(melee_expected_hp(wv, strength), ls="--", lw=1,
               label=f"one auto-attack (STR {strength})", color="tab:gray" if strength == 40 else "black")
mm.skill_axis(ax, "HP vs armor 0", "Gash roll range vs a normal swing (WV 40)")
mm.mark_caps(ax, extra=[(300, "knee")])
ax.legend(loc="upper left", bbox_to_anchor=(0.0, 0.85))

# Value per HP spent: damage dealt / self-cost, against caster current HP.
ax = axes[1]
for hp in (200, 400, 800, 1600):
    line, = ax.plot(P, gash_expected_hp(P, 40, armor=20) / gash_self_cost_hp(hp), label=f"caster at {hp} HP")
    if hp == 400:
        ax.plot(P, old_gash_expected_hp(P, 40, armor=20) / gash_self_cost_hp(hp), color=line.get_color(),
                ls=":", lw=1.2, label=f"before, caster at {hp} HP")
mm.skill_axis(ax, "target HP lost ÷ caster HP spent", "Damage dealt per HP spent (WV 40, armor 20)")
ax.axhline(1, color="grey", lw=1)
mm.mark_caps(ax, extra=[(300, "knee")])
ax.legend()
plt.tight_layout()

In [ ]:
# Gash vs a 10 s window of auto-attacks: how many swings is one Gash worth?
fig, ax = plt.subplots()
for wv in mm.WV_SWEEP:
    swing = melee_expected_hp(wv, 80, armor=20)
    line, = ax.plot(P, gash_expected_hp(P, wv, armor=20) / swing, label=f"WV {wv}")
    if wv == 40:
        ax.plot(P, old_gash_expected_hp(P, wv, armor=20) / swing, color=line.get_color(), ls=":", lw=1.2,
                label=f"before, WV {wv}")
mm.skill_axis(ax, "Gash ÷ one normal hit", "Gash measured in auto-attacks (STR 80, armor 20)")
mm.mark_caps(ax, extra=[(200, "old ceiling"), (300, "knee")])
ax.legend()

# Observations

- **Endgame.** With WV 40 and STR 80 against armor 20, one Gash deals 74.9 HP at modified skill
  275 and 79.9 HP at 300. That is 7.0 and 7.4 normal hits (one hit is 10.75 HP). At 300 the value
  is exactly the old ceiling.
- **Skill past 300 still pays.** The bonus grows +1% per point beyond 300 with no ceiling. Skill
  350 deals 84.9 HP (7.9 hits), 6% more than 300. Skill 450 deals 94.9 HP (8.8 hits). Before, every
  point past 200 was wasted.
- **The mid-game is weaker.** The new curve is at or below the old one everywhere up to 300. At
  the raise cap of 105 the bonus is +210% instead of +315%. In the example above Gash drops from
  51.4 to 40.9 HP (4.8 → 3.8 normal hits, −20%). The old 105 value is only reached at skill 158.
  Damage per HP spent at 400 caster HP drops from 5.1 to 4.1. This is the main cost of the change
  and is worth watching in play.
- **Armor is a flat tax.** Gash is type 0, so armor is subtracted once from the full hit. The
  hit is several times WV, so a given armor value removes a shrinking share of it. The
  $\text{rand}(3W)$ term is wide, so even WV 40 against armor 80 still averages 8.7 HP at the skill
  floor; a Gash rarely glances off completely.
- **Weapon value multiplies everything.** The bonus scales W, so +WV from Sun's Blessing is worth
  several times its face value on Gash. Gash has no hit roll, so Inner Strength's Weapon Skill
  bonus does not affect it.
- **Self-cost is proportional to current HP.** The ratio of damage dealt to HP spent is best
  when the caster is wounded. At 200 HP the caster pays 5 HP; at 1600 HP the caster pays 40 HP
  for the same blow. The "too wounded" refusal (`a_hp ≤ cost`) only triggers at 1 milli-HP, so
  Gash remains usable at very low HP.
- **Stale comments.** The `skill_gash` doc comment and the self-damage comment still describe the
  cost as "5% of current HP, halved by the talent buff". The code always charges a flat `a_hp/40`
  (2.5%).
- **Redundant guard.** `weapon` is already `.max(1)`, so the later `weapon.max(1) * 3` is a no-op.